# Residual Networks (ResNet)

> Based on Stanford CS230 — C4M2, Andrew Ng / DeepLearning.AI

He et al. (2015) showed that adding **skip connections** that short-circuit one or more layers allows training networks of 100–1000+ layers without degradation. ResNet won ImageNet 2015 with 3.57% top-5 error.

## Learning Objectives

1. Explain why very deep plain networks degrade despite more capacity
2. Implement a residual block with identity and projection shortcuts
3. Understand how skip connections enable gradient flow in very deep networks
4. Trace shapes through a ResNet-50 block

## The Residual Block

Instead of learning $H(x)$ directly, the block learns the **residual** $F(x) = H(x) - x$, so $H(x) = F(x) + x$:

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 640 160" width="640" height="160" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <defs>
    <marker id="arr3" markerWidth="7" markerHeight="7" refX="5" refY="3" orient="auto">
      <path d="M0,0 L0,6 L7,3 z" fill="#888"/>
    </marker>
  </defs>
  <!-- Main path -->
  <rect x="60"  y="55" width="90" height="50" rx="5" fill="#7ecba1" fill-opacity="0.15" stroke="#7ecba1" stroke-width="1.6"/>
  <rect x="220" y="55" width="90" height="50" rx="5" fill="#7ecba1" fill-opacity="0.15" stroke="#7ecba1" stroke-width="1.6"/>
  <rect x="380" y="65" width="36" height="30" rx="5" fill="#f4b942" fill-opacity="0.2"  stroke="#f4b942" stroke-width="1.6"/>
  <rect x="480" y="55" width="90" height="50" rx="5" fill="#5b9bd5" fill-opacity="0.12" stroke="#5b9bd5" stroke-width="1.6"/>
  <!-- Labels in blocks -->
  <text x="105" y="76" text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">Conv + BN</text>
  <text x="105" y="91" text-anchor="middle" fill="#3d7a5e" font-size="10">+ ReLU</text>
  <text x="265" y="76" text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">Conv + BN</text>
  <text x="265" y="91" text-anchor="middle" fill="#3d7a5e" font-size="10">(no ReLU yet)</text>
  <text x="398" y="84" text-anchor="middle" fill="#a07010" font-size="16" font-weight="bold">+</text>
  <text x="525" y="76" text-anchor="middle" fill="#3a7bbf" font-size="11" font-weight="bold">ReLU</text>
  <text x="525" y="91" text-anchor="middle" fill="#3a7bbf" font-size="10">A[l+2]</text>
  <!-- Main path arrows -->
  <line x1="20"  y1="80" x2="57"   y2="80" stroke="#888" stroke-width="1.4" marker-end="url(#arr3)"/>
  <line x1="150" y1="80" x2="217"  y2="80" stroke="#888" stroke-width="1.4" marker-end="url(#arr3)"/>
  <line x1="310" y1="80" x2="377"  y2="80" stroke="#888" stroke-width="1.4" marker-end="url(#arr3)"/>
  <line x1="416" y1="80" x2="477"  y2="80" stroke="#888" stroke-width="1.4" marker-end="url(#arr3)"/>
  <line x1="570" y1="80" x2="620"  y2="80" stroke="#888" stroke-width="1.4" marker-end="url(#arr3)"/>
  <!-- Skip connection -->
  <path d="M 20 80 Q 20 130 398 130 Q 416 130 416 80" fill="none" stroke="#e05c5c" stroke-width="2" stroke-dasharray="6,3" marker-end="url(#arr3)"/>
  <!-- x label at start -->
  <text x="12"  y="74" text-anchor="middle" fill="#555" font-size="11" font-weight="bold">x</text>
  <!-- Skip label -->
  <text x="220" y="148" text-anchor="middle" fill="#e05c5c" font-size="10" font-weight="bold">Skip connection (identity shortcut)</text>
  <!-- F(x) label above main path -->
  <text x="220" y="48"  text-anchor="middle" fill="#3d7a5e" font-size="10" font-style="italic">F(x)  =  main path</text>
  <!-- Output label -->
  <text x="595" y="74" text-anchor="middle" fill="#555" font-size="11">H(x)</text>
  <text x="620" y="86" text-anchor="middle" fill="#888" font-size="9">=F(x)+x</text>
</svg>

If the optimal transformation is close to identity, the network just learns $F(x) \approx 0$ — much easier than learning $H(x) \approx x$ from scratch.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# ---- Gradient flow simulation: plain vs residual networks ----
def relu(z): return np.maximum(0, z)
def relu_d(z): return (z > 0).astype(float)

def simulate_grad_flow(n_layers, use_skip, scale=0.5, seed=1):
    """Simulate gradient norms through layers (no real training — illustrative)."""
    np.random.seed(seed)
    grad_norms = [1.0]
    g = 1.0
    for l in range(n_layers):
        w_grad = scale * np.abs(np.random.randn())
        if use_skip and l % 2 == 1:
            g = g * w_grad + grad_norms[-2]  # skip adds previous gradient
        else:
            g = g * w_grad
        grad_norms.append(g)
    return grad_norms

depths = [20, 50, 100]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Gradient Flow: Plain vs Residual Networks', fontsize=13, fontweight='bold')

for ax, depth in zip(axes, depths):
    plain_grads = simulate_grad_flow(depth, use_skip=False, scale=0.85)
    resnet_grads = simulate_grad_flow(depth, use_skip=True,  scale=0.85)
    ax.semilogy(range(depth+1), plain_grads,  color=P[1], lw=2, label='Plain Network')
    ax.semilogy(range(depth+1), resnet_grads, color=P[3], lw=2, label='ResNet (skip connections)')
    ax.axhline(1e-3, color='#bbb', lw=1, ls='--', label='Vanishing threshold')
    ax.set_xlabel('Layer (from output → input)')
    ax.set_ylabel('Gradient norm (log)')
    ax.set_title(f'{depth}-layer network')
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.show()

# ---- Bottleneck block (ResNet-50 style) shape trace ----
def conv_out(n, f, p, s): return (n + 2*p - f) // s + 1

def resnet50_bottleneck(n_in, c_in, c_mid, c_out, stride=1):
    """1×1 → 3×3 → 1×1 bottleneck. Returns (n_out, c_out, params)."""
    # 1×1 conv: reduce channels
    p1 = (1*1*c_in + 1) * c_mid
    n1 = conv_out(n_in, 1, 0, 1)
    # 3×3 conv (may halve spatial dims)
    p2 = (3*3*c_mid + 1) * c_mid
    n2 = conv_out(n1, 3, 1, stride)
    # 1×1 conv: expand channels
    p3 = (1*1*c_mid + 1) * c_out
    n3 = conv_out(n2, 1, 0, 1)
    # projection shortcut (if shapes differ)
    proj = (1*1*c_in + 1)*c_out if (c_in != c_out or stride != 1) else 0
    total = p1 + p2 + p3 + proj
    return n3, c_out, total

print("ResNet-50 Bottleneck Block Trace (first two stages):")
print(f"{'Block':30s} {'H':>4} {'W':>4} {'C':>5} {'Params':>10}")
print("-" * 60)

n, c = 56, 64
for i, (c_mid, c_out, s) in enumerate([(64,256,1),(64,256,1),(64,256,1)], 1):
    c_in = 64 if i == 1 else 256
    n, c, p = resnet50_bottleneck(n, c_in, c_mid, c_out, stride=s)
    print(f"Stage2 Block {i}: 1×1→3×3→1×1 {n:>4} {n:>4} {c:>5} {p:>10,}")

for i, (c_mid, c_out, s) in enumerate([(128,512,2),(128,512,1),(128,512,1),(128,512,1)], 1):
    c_in = 256 if i == 1 else 512
    n, c, p = resnet50_bottleneck(n, c_in, c_mid, c_out, stride=s)
    print(f"Stage3 Block {i}: 1×1→3×3→1×1 {n:>4} {n:>4} {c:>5} {p:>10,}")
